# 02 · Calibración del K-Means: qué receta congela `configs/kmeans.json`

**Laboratorio en Python, no sistema.** Go no decide features, transformación, nivel ni k: los lee de
`configs/kmeans.json`. Este notebook corre la **grilla de la spec de calibración (§3)** con
scikit-learn sobre una submuestra sistemática del silver y aplica la **regla de elección** en su
orden, para proponer esa receta con sus cifras.

Grilla: features F1 {PM2.5, PM10} · F2 {PM10, fracción fina} × meteorología M0 · M1 (viento,
humedad) · M2 (M1 + temperatura, presión) × transformación T1 crudo · T2 log1p en contaminantes ×
nivel N1 absoluto · N2 por estación × k = 2…10 × 5 semillas (`KMeans(init="k-means++")`).

Por corrida: inercia, silueta (10 000 puntos), ARI entre semillas, ARI por remuestreo en bloques de
días, pureza y recall de la etiqueta ECA en los regímenes que exceden, NMI con estación, hora y mes,
tamaño del cluster más chico y distancia mínima entre centroides.

| Entra | Sale |
|---|---|
| `data/silver/oefa_ancho_prototipo.csv` (notebook 01) o `oefa_ancho.csv` (Go) | `reports/calibracion.csv` (una fila por corrida) y `calibracion_resumen.csv` |
| | `reports/calibracion_decision.md` (tabla de decisión) |
| | `reports/kmeans_propuesto.json` (mismo esquema que `configs/kmeans.json`) |
| | `reports/centroides_sklearn.json` (referencia para el test de equivalencia con Go) |

La propuesta **no** se copia sola a `configs/kmeans.json`: lo decide Rody mirando la tabla.

## 0. Entorno

In [ ]:
# Colab trae casi todo; esta celda instala solo lo que falte (fuera de Colab no hace nada si ya está).
import importlib.util, subprocess, sys
PAQUETES = {"scikit-learn": "sklearn", "joblib": "joblib", "scipy": "scipy", "pandas": "pandas", "matplotlib": "matplotlib"}
faltan = [pip for pip, mod in PAQUETES.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", faltan)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)

In [ ]:
# ── Entorno: Drive, raíz de las capas y código compartido (notebooks/aire_lab.py) ──────────────
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:  # fuera de Colab no se monta nada y la celda no falla
    from google.colab import drive
    drive.mount("/content/drive")

# Parámetros de ubicación: lo único que hay que tocar.
RAIZ_DRIVE = Path("/content/drive/MyDrive/aire-distribuido")  # capas medallón en Drive
RAIZ_LOCAL = os.environ.get("AIRE_RAIZ")   # alternativa local (p. ej. el sintético); None = repo
REPO_GIT = "https://github.com/R0SEWT/aire-distribuido.git"
RAMAS = ["feature/spec-pc3", "develop"]   # de dónde clonar aire_lab.py en Colab, en ese orden
ACTUALIZAR_CODIGO = False                  # True: vuelve a clonar (si cambió aire_lab.py)


def ubicar_repo() -> Path:
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / "notebooks" / "aire_lab.py").exists():
            return c
    destino = Path("/content/aire-distribuido")
    if ACTUALIZAR_CODIGO:
        subprocess.run(["rm", "-rf", str(destino)], check=False)
    for rama in RAMAS:
        if (destino / "notebooks" / "aire_lab.py").exists():
            break
        subprocess.run(["rm", "-rf", str(destino)], check=False)
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", rama, REPO_GIT, str(destino)],
                       check=False)
    if not (destino / "notebooks" / "aire_lab.py").exists():
        raise FileNotFoundError("No encuentro notebooks/aire_lab.py: clona el repo o súbelo junto al notebook.")
    return destino


REPO = ubicar_repo()
sys.path.insert(0, str(REPO / "notebooks"))
RAIZ = Path(RAIZ_LOCAL) if RAIZ_LOCAL else (RAIZ_DRIVE if EN_COLAB else REPO)
DATA, REPORTS = RAIZ / "data", RAIZ / "reports"
CONFIGS = [RAIZ / "configs", REPO / "configs"]  # la de la raíz (Drive), si existe, va primero
# Figuras del informe: en local, docs/informe/generado/figuras/; en Colab, reports/figuras/ de Drive.
FIGURAS = Path(os.environ["AIRE_FIGURAS"]) if os.environ.get("AIRE_FIGURAS") else (
    RAIZ / "reports" / "figuras" if EN_COLAB else REPO / "docs" / "informe" / "generado" / "figuras")
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

import importlib
import aire_lab as al
importlib.reload(al)
CMAPS = al.estilo_figuras()
SINTETICO = al.es_sintetico(RAIZ)  # se vuelve a mirar al leer los datos (estaciones SINT-/SIN-)
print(f"Colab: {EN_COLAB}\nrepo: {REPO}\nraíz de las capas: {RAIZ}\nfiguras del informe: {FIGURAS}")

In [ ]:
import json, math, time, itertools, warnings, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import sklearn
from sklearn.cluster import KMeans

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)
FIG_CAL = REPORTS / "figuras_calibracion"
FIG_CAL.mkdir(parents=True, exist_ok=True)


def guardar(fig, nombre):
    al.marca_sintetico(fig, SINTETICO)
    fig.savefig(FIG_CAL / nombre, dpi=150)
    plt.show()

## 1. Parámetros

Los umbrales son los de la spec (§3), puntos de partida que se reportan tal cual.
`K_DESDE_CODO` es la lectura de «el k más chico que cumpla lo anterior, **mirando el codo**»: el codo
de cada variante es el piso de k, para no elegir un k = 2 que parte el aire en «bajo» y «alto»
**[decisión mía, reversible]**.

In [ ]:
FUENTE_SILVER = None      # None: prototipo del notebook 01 si existe, si no el silver de Go; o una ruta
N_SUBMUESTRA = 200_000    # filas anchas de la submuestra sistemática (cada n-ésima)
N_SILUETA = 10_000        # la silueta es O(n²)
KS = list(range(2, 11))
SEMILLAS = [11, 23, 42, 57, 89]
N_JOBS = -1               # joblib: todos los núcleos
K_DESDE_CODO = True
GASES_EN_FEATURES = []    # p. ej. ["so2"] si el EDA dice que la gran mayoría de estaciones lo mide (spec §2.5)

UMBRALES = {
    "ari_semillas": 0.8,  # (1) estable
    "ari_dias": 0.7,
    "pureza": 0.6,        # (2) separa el ECA
    "recall": 0.5,
    "min_cluster": 0.01,  # (4) interpretable: ningún cluster bajo el 1 %
    "sep_linf": 1.0,      #     y cada par de centroides difiere en > 1 desviación en alguna variable
}
# Lo que la grilla no calibra: se copia de configs/kmeans.json (Go los usa tal cual).
FIJOS = {"semilla": 42, "tolerancia": 1e-6, "max_iter": 100, "workers": 0, "chunk": 10000}
try:
    _actual, _ = al.leer_receta([c / "kmeans.json" for c in CONFIGS])
    FIJOS = {k: _actual.get(k, v) for k, v in FIJOS.items()}
except FileNotFoundError:
    pass

FAMILIA_F = {"F1": ["pm2_5", "pm10"], "F2": ["pm10", "fraccion_fina"]}
FAMILIA_M = {"M0": [], "M1": ["viento_velocidad", "humedad"],
             "M2": ["viento_velocidad", "humedad", "temperatura", "presion"]}
FAMILIA_T = {"T1": False, "T2": True}
FAMILIA_N = {"N1": "absoluto", "N2": "estacion"}

## 2. Silver y submuestra sistemática

In [ ]:
candidatos = [DATA / "silver" / "oefa_ancho_prototipo.csv", DATA / "silver" / "oefa_ancho.csv"]
RUTA_SILVER = Path(FUENTE_SILVER) if FUENTE_SILVER else next((c for c in candidatos if c.exists()), None)
if RUTA_SILVER is None or not RUTA_SILVER.exists():
    raise FileNotFoundError(f"No hay silver en {candidatos}: corre el notebook 01 o el cargador de Go.")
silver = al.leer_csv_contrato(RUTA_SILVER).sort_values(["estacion", "fecha_hora"], kind="mergesort").reset_index(drop=True)
SHA_SILVER = al.sha256(RUTA_SILVER)
SINTETICO = SINTETICO or al.es_sintetico(estaciones=silver["estacion"].unique())
paso = max(1, math.ceil(len(silver) / N_SUBMUESTRA))
sub = silver.iloc[::paso].reset_index(drop=True)
print(f"Silver: {RUTA_SILVER.name} · {len(silver):,} filas anchas · {silver['estacion'].nunique()} estaciones")
print(f"Submuestra sistemática: cada {paso}.ª fila → {len(sub):,} filas")

# Metadatos de cada fila de la submuestra: no son features, sirven para medir.
cod_est = {e: i for i, e in enumerate(sorted(sub["estacion"].unique()))}
META = pd.DataFrame({
    "est": sub["estacion"].map(cod_est).to_numpy(),
    "hora": sub["fecha_hora"].dt.hour.to_numpy(),
    "mes": sub["fecha_hora"].dt.month.to_numpy(),
    "dia": (sub["fecha_hora"].dt.normalize() - pd.Timestamp("2000-01-01")).dt.days.to_numpy(),
    "eca": al.excede_horario(sub).to_numpy(),
    "pm2_5": sub["pm2_5"].to_numpy() if "pm2_5" in sub else np.nan,
    "pm10": sub["pm10"].to_numpy() if "pm10" in sub else np.nan,
})
print(f"Horas sobre el ECA en la submuestra: {100 * META['eca'].mean():.2f} %")

## 3. Las 24 variantes

Cada variante usa sus filas completas (regla 8): M2 pierde las estaciones sin presión, F1 y F2 las
que no miden las dos PM. La tabla dice cuántas filas y estaciones quedan en cada una; a igualdad de
lo demás se prefiere la que cubre más estaciones.

In [ ]:
VARIANTES = {}
for f, m, t, n in itertools.product(FAMILIA_F, FAMILIA_M, FAMILIA_T, FAMILIA_N):
    feats = FAMILIA_F[f] + GASES_EN_FEATURES + FAMILIA_M[m]
    logs = [x for x in feats if x in al.CONTAMINANTES] if FAMILIA_T[t] else []
    receta = {"features": feats, "log1p": logs, "nivel": FAMILIA_N[n]}
    mascara = al.completas(sub, feats, logs).to_numpy()
    nombre = f"{f}-{m}-{t}-{n}"
    if mascara.sum() < 50 * max(KS):
        print(f"{nombre}: solo {mascara.sum()} filas completas, se omite")
        continue
    gold_v, stats_v = al.construir_gold(sub.loc[mascara].reset_index(drop=True), receta)
    VARIANTES[nombre] = {"F": f, "M": m, "T": t, "N": n, "receta": receta, "mascara": mascara,
                         "X": np.ascontiguousarray(gold_v[feats].to_numpy(dtype=float))}
tabla_var = pd.DataFrame([{"variante": k, "features": " + ".join(v["receta"]["features"]),
                           "log1p": " + ".join(v["receta"]["log1p"]) or "—", "nivel": v["receta"]["nivel"],
                           "filas": int(v["mascara"].sum()),
                           "estaciones": int(META.loc[v["mascara"], "est"].nunique())}
                          for k, v in VARIANTES.items()])
display(tabla_var)

## 4. La grilla

Una tarea por (variante, k) con sus cinco semillas: así el ARI entre semillas se calcula dentro de
la tarea. El remuestreo es por **bloques de días** (días con reemplazo, con todas sus horas), porque
las horas consecutivas no son independientes: ajustar sobre el remuestreo y predecir la submuestra
completa da el ARI por días. Un régimen «excede el ECA» si su PM2.5 medio supera 50 µg/m³ o su PM10
medio supera 100 µg/m³ (spec PC3, §6.3); pureza y recall se miden contra la etiqueta horaria.

In [ ]:
def evaluar(nombre, k, X, est, hora, mes, dia, eca, pm25, pm10, semillas, n_silueta):
    """Corre las semillas de una (variante, k) y devuelve una fila por corrida."""
    import time as _t
    import warnings as _w
    import numpy as _np
    from sklearn.cluster import KMeans as _KM
    from sklearn.metrics import adjusted_rand_score as _ari, normalized_mutual_info_score as _nmi, silhouette_score as _sil
    _w.filterwarnings("ignore")
    n = len(X)
    orden = _np.argsort(dia, kind="stable")
    _, ini = _np.unique(dia[orden], return_index=True)
    bloques = _np.split(orden, ini[1:])
    filas, etiquetas = [], []
    for s in semillas:
        t0 = _t.perf_counter()
        km = _KM(n_clusters=k, init="k-means++", n_init=1, random_state=s).fit(X)
        lab = km.labels_
        C = km.cluster_centers_
        sil = float(_sil(X, lab, sample_size=min(n_silueta, n), random_state=0)) if len(_np.unique(lab)) > 1 else _np.nan
        rng = _np.random.default_rng(s)
        idx = _np.concatenate([bloques[i] for i in rng.integers(0, len(bloques), len(bloques))])
        km_b = _KM(n_clusters=k, init="k-means++", n_init=1, random_state=s).fit(X[idx])
        ari_d = float(_ari(lab, km_b.predict(X)))
        medias = {}
        for nom, v in (("pm2_5", pm25), ("pm10", pm10)):
            ok = _np.isfinite(v)
            cnt = _np.bincount(lab[ok], minlength=k)
            sm = _np.bincount(lab[ok], weights=v[ok], minlength=k)
            medias[nom] = _np.where(cnt > 0, sm / _np.maximum(cnt, 1), _np.nan)
        excede = (_np.nan_to_num(medias["pm2_5"]) > 50) | (_np.nan_to_num(medias["pm10"]) > 100)
        sel = excede[lab]
        p_eca_c = _np.bincount(lab, weights=eca.astype(float), minlength=k) / _np.maximum(_np.bincount(lab, minlength=k), 1)
        tam = _np.bincount(lab, minlength=k) / n
        dif = C[:, None, :] - C[None, :, :]
        tri = _np.triu_indices(k, 1)
        filas.append({
            "variante": nombre, "k": k, "semilla": s, "n_filas": n, "n_estaciones": int(len(_np.unique(est))),
            "inercia": float(km.inertia_), "iteraciones": int(km.n_iter_), "silueta": sil, "ari_dias": ari_d,
            "pureza": float(eca[sel].mean()) if sel.any() else _np.nan,
            "recall": float(sel[eca].mean()) if eca.any() else _np.nan,
            "frac_excede": float(sel.mean()), "n_regimenes_excede": int(excede.sum()),
            "enriquecimiento_max": float(p_eca_c.max() / eca.mean()) if eca.any() else _np.nan,
            "nmi_estacion": float(_nmi(lab, est)), "nmi_hora": float(_nmi(lab, hora)),
            "nmi_mes": float(_nmi(lab, mes)), "nmi_eca": float(_nmi(lab, eca)),
            "min_cluster_frac": float(tam.min()),
            "dist_min_centroides": float(_np.sqrt((dif ** 2).sum(-1))[tri].min()),
            "sep_linf": float(_np.abs(dif).max(-1)[tri].min()),
            "segundos": _t.perf_counter() - t0,
        })
        etiquetas.append(lab)
    for i, f in enumerate(filas):
        f["ari_semillas"] = float(_np.mean([_ari(etiquetas[i], etiquetas[j]) for j in range(len(etiquetas)) if j != i]))
    return filas


tareas = []
for nombre, V in VARIANTES.items():
    mm = META[V["mascara"]]
    args = (V["X"], mm["est"].to_numpy(), mm["hora"].to_numpy(), mm["mes"].to_numpy(), mm["dia"].to_numpy(),
            mm["eca"].to_numpy(), mm["pm2_5"].to_numpy(dtype=float), mm["pm10"].to_numpy(dtype=float))
    for k in sorted(KS, reverse=True):  # los k grandes primero: reparten mejor la carga
        tareas.append((nombre, k) + args)
print(f"{len(tareas)} tareas × {len(SEMILLAS)} semillas = {len(tareas) * len(SEMILLAS)} corridas")

t0 = time.perf_counter()
try:
    from joblib import Parallel, delayed
    salidas = Parallel(n_jobs=N_JOBS, verbose=2)(delayed(evaluar)(*t, SEMILLAS, N_SILUETA) for t in tareas)
except ImportError:
    print("Sin joblib: la grilla corre en serie")
    salidas = [evaluar(*t, SEMILLAS, N_SILUETA) for t in tareas]
print(f"Grilla en {(time.perf_counter() - t0) / 60:.1f} min")

cal = pd.DataFrame([f for s in salidas for f in s])
for col in ["F", "M", "T", "N"]:
    cal[col] = cal["variante"].map({k: v[col] for k, v in VARIANTES.items()})
cal["features"] = cal["variante"].map({k: "+".join(v["receta"]["features"]) for k, v in VARIANTES.items()})
cal["log1p"] = cal["variante"].map({k: "+".join(v["receta"]["log1p"]) for k, v in VARIANTES.items()})
cal["nivel"] = cal["variante"].map({k: v["receta"]["nivel"] for k, v in VARIANTES.items()})
cal = cal.sort_values(["variante", "k", "semilla"]).reset_index(drop=True)
cal.to_csv(REPORTS / "calibracion.csv", index=False)
print("Escrito", REPORTS / "calibracion.csv", cal.shape)

## 5. Regla de elección

Promedio de las cinco semillas por (variante, k), y la regla de la spec **en este orden**:
(1) estable: ARI entre semillas > 0,8 y por días > 0,7; (2) separa el ECA: pureza > 0,6 y recall
> 0,5 en los regímenes que exceden; (3) no es geografía: NMI con estación < NMI con la etiqueta
ECA (si el absoluto falla aquí, gana el nivel por estación); (4) interpretable: ningún cluster bajo
el 1 % y centroides distinguibles; (5) el k más chico que cumpla, desde el codo de la variante.
Si nadie cumple las cuatro, se elige entre quienes cumplen el prefijo más largo y se dice qué
umbral no cumplió nadie. Desempates a igual k: más estaciones cubiertas, mejor pureza × recall,
mejor silueta.

In [ ]:
res = (cal.assign(pureza0=cal["pureza"].fillna(0), recall0=cal["recall"].fillna(0))
       .groupby(["variante", "F", "M", "T", "N", "k"], as_index=False)
       .agg(n_filas=("n_filas", "first"), n_estaciones=("n_estaciones", "first"),
            inercia=("inercia", "mean"), silueta=("silueta", "mean"),
            ari_semillas=("ari_semillas", "mean"), ari_dias=("ari_dias", "mean"),
            pureza=("pureza0", "mean"), recall=("recall0", "mean"), frac_excede=("frac_excede", "mean"),
            nmi_estacion=("nmi_estacion", "mean"), nmi_hora=("nmi_hora", "mean"), nmi_mes=("nmi_mes", "mean"),
            nmi_eca=("nmi_eca", "mean"), min_cluster=("min_cluster_frac", "min"),
            sep_linf=("sep_linf", "mean"), dist_min=("dist_min_centroides", "mean")))
codos = res.groupby("variante").apply(lambda g: al.codo(g["k"], g["inercia"]), include_groups=False).rename("codo")
res = res.merge(codos, on="variante")
res["f_eca"] = res["pureza"] * res["recall"]
res["c1_estable"] = (res["ari_semillas"] > UMBRALES["ari_semillas"]) & (res["ari_dias"] > UMBRALES["ari_dias"])
res["c2_eca"] = (res["pureza"] > UMBRALES["pureza"]) & (res["recall"] > UMBRALES["recall"])
res["c3_no_geografia"] = res["nmi_estacion"] < res["nmi_eca"]
res["c4_interpretable"] = (res["min_cluster"] >= UMBRALES["min_cluster"]) & (res["sep_linf"] > UMBRALES["sep_linf"])
res["k_desde_codo"] = res["k"] >= res["codo"] if K_DESDE_CODO else True
CRIT = ["c1_estable", "c2_eca", "c3_no_geografia", "c4_interpretable"]
res["prefijo"] = 0
for i in range(len(CRIT)):
    res.loc[res[CRIT[: i + 1]].all(axis=1), "prefijo"] = i + 1
res.to_csv(REPORTS / "calibracion_resumen.csv", index=False)

ORDEN = ["k", "n_estaciones", "f_eca", "silueta"]
ASC = [True, False, False, False]
pool = res[res["k_desde_codo"]]
alcanzado = int(pool["prefijo"].max())
cand = pool[pool["prefijo"] == alcanzado].sort_values(ORDEN, ascending=ASC)
elegida = cand.iloc[0]
otras = cand[cand["variante"] != elegida["variante"]]
if len(otras):
    segunda = otras.iloc[0]
else:
    resto = pool[pool["variante"] != elegida["variante"]].sort_values(["prefijo"] + ORDEN, ascending=[False] + ASC)
    segunda = resto.iloc[0] if len(resto) else None
prov = res[(res["variante"] == "F2-M1-T2-N1") & (res["k"] == 5)]
provisional = prov.iloc[0] if len(prov) else None

cuantos = pd.DataFrame({
    "criterio": CRIT,
    "cumplen solos": [int(pool[c].sum()) for c in CRIT],
    "cumplen con los anteriores": [int((pool["prefijo"] >= i + 1).sum()) for i in range(len(CRIT))],
    "de": len(pool),
})
display(cuantos)
print(f"Elegida: {elegida['variante']} con k = {int(elegida['k'])} (cumple {alcanzado} de 4 criterios en orden)")
if segunda is not None:
    print(f"Segunda: {segunda['variante']} con k = {int(segunda['k'])}")
cols_ver = ["variante", "k", "codo", "n_estaciones", "ari_semillas", "ari_dias", "pureza", "recall", "nmi_estacion",
            "nmi_eca", "min_cluster", "sep_linf", "silueta", "prefijo"]
display(cand[cols_ver].head(10).round(3))

In [ ]:
# Mapa de la grilla: cuántos criterios cumple, en orden, cada (variante, k).
piv = res.pivot(index="variante", columns="k", values="prefijo").reindex(sorted(VARIANTES))
fig, ax = plt.subplots(figsize=(8.5, 0.32 * len(piv) + 1.6))
from matplotlib.colors import ListedColormap, BoundaryNorm
cmap = ListedColormap(["#f0efec", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab"])
im = ax.imshow(piv.to_numpy(), cmap=cmap, norm=BoundaryNorm(np.arange(-0.5, 5.5), 5), aspect="auto")
ax.set_xticks(range(len(piv.columns)), piv.columns)
ax.set_yticks(range(len(piv.index)), piv.index, fontsize=8)
ax.grid(False)
for i, var in enumerate(piv.index):
    c = codos.get(var)
    if c in list(piv.columns):
        ax.plot(list(piv.columns).index(c), i, marker="|", color=al.TINTA, markersize=10)
fi, fj = list(piv.index).index(elegida["variante"]), list(piv.columns).index(int(elegida["k"]))
ax.add_patch(plt.Rectangle((fj - 0.5, fi - 0.5), 1, 1, fill=False, edgecolor=al.CRITICO, linewidth=2))
cb = fig.colorbar(im, ax=ax, ticks=range(5), shrink=0.7)
cb.set_label("criterios cumplidos en orden (1 estable, 2 ECA, 3 no geografía, 4 interpretable)", fontsize=8)
ax.set_xlabel("k   (| = codo de la variante; recuadro = elegida)")
ax.set_title("Calibración: cuántos criterios cumple cada variante y k", loc="left")
guardar(fig, "calibracion_grilla.png")

g = res[res["variante"] == elegida["variante"]].sort_values("k")
fig, axs = plt.subplots(1, 3, figsize=(12, 3.3))
axs[0].plot(g["k"], g["inercia"], marker="o", color=al.PALETA[0])
axs[0].set_title("Inercia (media de 5 semillas)", loc="left")
axs[1].plot(g["k"], g["silueta"], marker="o", color=al.PALETA[0])
axs[1].set_title("Silueta", loc="left")
axs[2].plot(g["k"], g["ari_semillas"], marker="o", color=al.PALETA[0], label="ARI entre semillas")
axs[2].plot(g["k"], g["ari_dias"], marker="o", color=al.PALETA[1], label="ARI por días")
axs[2].axhline(UMBRALES["ari_dias"], color=al.TINTA_2, linewidth=0.8)
axs[2].set_title("Estabilidad", loc="left")
axs[2].legend()
for ax in axs:
    ax.axvline(elegida["k"], color=al.CRITICO, linewidth=1, linestyle="--")
    ax.set_xlabel("k")
fig.suptitle(f"Variante elegida {elegida['variante']}", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
guardar(fig, "calibracion_elegida.png")

## 6. Receta propuesta y centroides de referencia

La receta elegida se ajusta sobre **todo** el silver (no la submuestra), con las cinco semillas, y se
guarda la de menor inercia. Los centroides se ordenan por la primera feature (ascendente) para que
el orden no dependa de la numeración de scikit-learn; el test de Go los compara con asignación
húngara y tolerancia.

In [ ]:
receta = dict(VARIANTES[elegida["variante"]]["receta"])
propuesta = {"features": receta["features"], "log1p": receta["log1p"], "nivel": receta["nivel"],
             "k": int(elegida["k"]), **FIJOS}
(REPORTS / "kmeans_propuesto.json").write_text(json.dumps(propuesta, indent=2) + "\n", encoding="utf-8")
print(json.dumps(propuesta, indent=2))

gold_full, stats_full = al.construir_gold(silver, receta)
Xf = gold_full[receta["features"]].to_numpy(dtype=float)
mejor = None
for s in SEMILLAS:
    km = KMeans(n_clusters=propuesta["k"], init="k-means++", n_init=1, random_state=s).fit(Xf)
    if mejor is None or km.inertia_ < mejor[1].inertia_:
        mejor = (s, km)
s_mejor, km = mejor
orden_c = np.lexsort(km.cluster_centers_.T[::-1])
Cz = km.cluster_centers_[orden_c]
lab = np.argsort(orden_c)[km.labels_]
Co = al.desestandarizar(Cz, stats_full)
ref = {
    "receta": propuesta, "fuente_silver": RUTA_SILVER.name, "sha256_silver": SHA_SILVER,
    "n": int(len(Xf)), "media": stats_full["media"], "desv": stats_full["desv"],
    "sklearn": sklearn.__version__, "semilla_sklearn": int(s_mejor), "n_semillas": len(SEMILLAS),
    "inercia": float(km.inertia_), "iteraciones": int(km.n_iter_),
    "orden_centroides": "por la primera feature, ascendente",
    "tamanos": np.bincount(lab, minlength=propuesta["k"]).tolist(),
    "centroides_estandarizados": Cz.tolist(),
    "centroides_desestandarizados": None if Co is None else Co.tolist(),
    "origen": "sintetico" if SINTETICO else "oefa",
}
(REPORTS / "centroides_sklearn.json").write_text(json.dumps(ref, indent=2) + "\n", encoding="utf-8")
print(f"Centroides de referencia: {len(Xf):,} filas, inercia {km.inertia_:.4f}, semilla {s_mejor}")
display(pd.DataFrame(Cz, columns=receta["features"]).round(3))
if Co is not None:
    display(pd.DataFrame(Co, columns=receta["features"]).round(3))

## 7. `reports/calibracion_decision.md`

In [ ]:
NOMBRE_C = {"c1_estable": "(1) estable", "c2_eca": "(2) separa el ECA", "c3_no_geografia": "(3) no es geografía",
            "c4_interpretable": "(4) interpretable"}
METRICAS = [("k", "k", "{:.0f}"), ("codo", "codo de la variante", "{:.0f}"), ("n_filas", "filas (submuestra)", "{:,.0f}"),
            ("n_estaciones", "estaciones", "{:.0f}"), ("ari_semillas", "ARI entre semillas", "{:.3f}"),
            ("ari_dias", "ARI por días", "{:.3f}"), ("pureza", "pureza «excede»", "{:.3f}"), ("recall", "recall «excede»", "{:.3f}"),
            ("frac_excede", "fracción de horas en «excede»", "{:.3f}"), ("nmi_estacion", "NMI estación", "{:.3f}"),
            ("nmi_eca", "NMI etiqueta ECA", "{:.3f}"), ("nmi_hora", "NMI hora", "{:.3f}"), ("nmi_mes", "NMI mes", "{:.3f}"),
            ("min_cluster", "cluster más chico", "{:.3f}"), ("sep_linf", "separación L∞ mínima", "{:.2f}"),
            ("silueta", "silueta", "{:.3f}"), ("inercia", "inercia", "{:,.1f}")]


def col(r):
    if r is None:
        return ["—"] * (len(METRICAS) + len(CRIT) + 3)
    rec = VARIANTES[r["variante"]]["receta"]
    out = [f"`{r['variante']}`", ", ".join(rec["features"]), ", ".join(rec["log1p"]) or "ninguna"]
    out += ["sí" if r[c] else "no" for c in CRIT]
    out += [f.format(r[m]).replace(",", " ") for m, _, f in METRICAS]
    return out


filas_md = ["variante", "features", "log1p"] + [NOMBRE_C[c] for c in CRIT] + [n for _, n, _ in METRICAS]
tabla_dec = pd.DataFrame({"": filas_md, "elegida": col(elegida), "segunda": col(segunda),
                          "provisional (F2-M1-T2-N1, k = 5)": col(provisional)})

def por_que(r):
    partes = [f"cumple {alcanzado} de los 4 criterios en orden"]
    if alcanzado == 4:
        partes.append("es el k más chico que los cumple" + (f" desde el codo (k = {int(r['codo'])})" if K_DESDE_CODO else ""))
    partes.append(f"cubre {int(r['n_estaciones'])} estaciones")
    partes.append(f"pureza × recall = {r['f_eca']:.3f}, silueta {r['silueta']:.3f}")
    return "; ".join(partes)

def cede(r, s):
    if s is None:
        return "no hay segunda variante con la que comparar."
    mejores = []
    for m, n, f in METRICAS[3:]:
        a, b = r[m], s[m]
        mayor_mejor = m != "nmi_estacion"
        if m in ("inercia", "frac_excede", "nmi_hora", "nmi_mes"):  # descriptivas: ni mejor ni peor
            continue
        if (b > a + 1e-9) if mayor_mejor else (b < a - 1e-9):
            mejores.append(f"{n} ({f.format(b)} frente a {f.format(a)})")
    crit_s = [NOMBRE_C[c] for c in CRIT if s[c] and not r[c]]
    texto = (f"la segunda (`{s['variante']}`, k = {int(s['k'])}) es mejor en: " + ("; ".join(mejores) if mejores else "nada medible"))
    if crit_s:
        texto += f". Cumple además {', '.join(crit_s)}"
    razon = ("tiene un k mayor" if s["k"] > r["k"] else "cubre menos estaciones" if s["n_estaciones"] < r["n_estaciones"]
             else "tiene menor pureza × recall" if s["f_eca"] < r["f_eca"] else "tiene menor silueta")
    if s["prefijo"] < alcanzado:
        razon = f"cumple menos criterios en orden ({int(s['prefijo'])} de 4)"
    return texto + f". Pierde porque {razon}."

nadie = [NOMBRE_C[c] + f" (umbral: {txt})" for c, txt in zip(CRIT, [
    f"ARI semillas > {UMBRALES['ari_semillas']} y por días > {UMBRALES['ari_dias']}",
    f"pureza > {UMBRALES['pureza']} y recall > {UMBRALES['recall']}",
    "NMI estación < NMI ECA",
    f"cluster más chico >= {UMBRALES['min_cluster']} y separación L∞ > {UMBRALES['sep_linf']}"]) if pool[c].sum() == 0]
cadena = (f"Ningún candidato cumple {NOMBRE_C[CRIT[alcanzado]]} junto con los anteriores." if alcanzado < 4 else
          "Al menos un candidato cumple los cuatro criterios.")

hoy = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
L = ["# Calibración del K-Means — tabla de decisión", "",
     f"> Generado por `notebooks/02_calibracion.ipynb` el {hoy}. Silver `{RUTA_SILVER.name}` (sha256 `{SHA_SILVER[:16]}…`), "
     f"{al.miles(len(silver))} filas anchas; submuestra sistemática cada {paso}.ª fila ({al.miles(len(sub))} filas). "
     f"{len(VARIANTES)} variantes × {len(KS)} valores de k × {len(SEMILLAS)} semillas = {len(cal)} corridas "
     f"(scikit-learn {sklearn.__version__}). Spec: `docs/specs/2026-10-31-kmeans-calibracion-design.md` §3."]
if SINTETICO:
    L += ["", "> **DATOS SINTÉTICOS.** La decisión de abajo prueba el notebook; no es la receta del proyecto."]
L += ["", "## Decisión", "", al.tabla_md(tabla_dec), "",
      f"**Variante elegida: `{elegida['variante']}` con k = {int(elegida['k'])}.** Por qué: {por_que(elegida)}.", "",
      f"**Qué cede frente a la segunda**: {cede(elegida, segunda)}", "",
      f"**Umbrales que no cumplió nadie** (desde el codo): " + ("; ".join(nadie) if nadie else "ninguno: cada criterio lo cumple al menos un candidato") + f". {cadena}", "",
      "Umbrales usados: " + ", ".join(f"{k} = {v}" for k, v in UMBRALES.items()) + f"; k desde el codo: {K_DESDE_CODO}.", "",
      "## Cuántos candidatos cumplen cada criterio", "", al.tabla_md(cuantos.assign(criterio=cuantos["criterio"].map(NOMBRE_C))), "",
      "## Mejor k de cada variante", "",
      "Para cada variante, el k más chico (desde su codo) con el prefijo de criterios más largo.", ""]
mejor_por_var = (pool.sort_values(["variante", "prefijo", "k"], ascending=[True, False, True]).groupby("variante").head(1)
                 .sort_values(["prefijo", "k", "n_estaciones"], ascending=[False, True, False]))
L += [al.tabla_md(mejor_por_var[["variante", "k", "codo", "prefijo", "n_estaciones", "ari_semillas", "ari_dias", "pureza",
                                 "recall", "nmi_estacion", "nmi_eca", "min_cluster", "sep_linf", "silueta"]].round(3), decimales=3), "",
      "## Receta propuesta (`reports/kmeans_propuesto.json`)", "",
      "Mismo esquema que `configs/kmeans.json`. `semilla`, `tolerancia`, `max_iter`, `workers` y `chunk` no los calibra "
      "la grilla: se copian del `configs/kmeans.json` actual. Copiar la propuesta allí es decisión de Rody.", "",
      "```json", json.dumps(propuesta, indent=2), "```", "",
      f"Centroides de scikit-learn sobre todo el silver ({al.miles(len(Xf))} filas, semilla {s_mejor}, inercia {km.inertia_:.4f}) "
      "en `reports/centroides_sklearn.json`, para el test de equivalencia con Go.", ""]
(REPORTS / "calibracion_decision.md").write_text("\n".join(L), encoding="utf-8")
print("Escrito", REPORTS / "calibracion_decision.md")
display(Markdown("\n".join(L[:16])))